# Universal Precision Runtime (UPR) — Notebook 02
## Level 2 (Layer-wise Activation Outputs) & Level 3 (Logit & Generation Evaluation)

---

### Objective
1. Enforce deterministic seed (`upr.set_seed(42)`).
2. Attach forward hooks (`LayerActivationCollector`) to all Transformer blocks.
3. Compare layer-wise hidden activations between baseline FP16 and 16-bit BitPlane model.
4. Evaluate output logits (Logit MAE, RMSE, Cosine Similarity with `compute_cosine_similarity`, KL Divergence).
5. Evaluate token-by-token greedy text generation.

In [ ]:
import os
import sys
import gc
import importlib
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Set Hugging Face Token safely from environment or Colab secrets
HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    if not HF_TOKEN:
        HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    pass

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/UniversalPrecisionRuntime'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    os.chdir(DRIVE_DIR)
except ImportError:
    pass

WORK_DIR = os.getcwd()
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

import upr
importlib.reload(upr)
upr.set_seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_ID = 'Qwen/Qwen3.5-0.8B'
BITPLANE_DIR = 'models/bitplane_qwen'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
eval_prompt = "Universal Precision Runtime provides dynamic multi-precision model execution from a single packed bit-plane checkpoint."
inputs = tokenizer(eval_prompt, return_tensors='pt').to(DEVICE)

### Step 2: Layer-wise Activation Hooking (Level 2)

In [ ]:
# Load original model
orig_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(DEVICE)
orig_collector = upr.LayerActivationCollector(orig_model)
orig_collector.register_hooks()

with torch.no_grad():
    orig_logits = orig_model(**inputs).logits

del orig_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Load 16-bit reconstructed model
recon_model = upr.BitPlaneModel.from_pretrained(BITPLANE_DIR, bits=16, device_map=DEVICE)
recon_collector = upr.LayerActivationCollector(recon_model)
recon_collector.register_hooks()

with torch.no_grad():
    recon_logits = recon_model(**inputs).logits

layer_metrics = upr.compare_layer_activations(orig_collector, recon_collector)
print(f"Captured forward activation metrics across {len(layer_metrics)} layer hook points.")

### Step 3: Logit & Text Generation Evaluation (Level 3)

In [ ]:
cos_sim = upr.compute_cosine_similarity(orig_logits, recon_logits)
kl_div = upr.compute_kl_divergence(orig_logits, recon_logits)
diff = torch.abs(orig_logits.to(torch.float32) - recon_logits.to(torch.float32))
mae = float(diff.mean().item())
rmse = float(torch.sqrt(torch.mean(diff ** 2)).item())

gen_prompt = "The core advantage of bit-plane representation is"
gen_inputs = tokenizer(gen_prompt, return_tensors='pt').to(DEVICE)

with torch.no_grad():
    gen_out = recon_model.generate(**gen_inputs, max_new_tokens=40, do_sample=False)
    gen_text = tokenizer.decode(gen_out[0], skip_special_tokens=True)

print('='*60)
print('LEVEL 3 LOGIT & GENERATION RESULTS')
print(f'Logit MAE:               {mae:.6e}')
print(f'Logit RMSE:              {rmse:.6e}')
print(f'Logit Cosine Similarity: {cos_sim:.6f}')
print(f'KL Divergence:           {kl_div:.6e}')
print('-'*60)
print(f'Generated Text: {gen_text}')
print('='*60)